# Data Manipulation

In order to get anything done, 
we need some way to store and manipulate data.
Generally, there are two important things 
we need to do with data: 
(i) acquire them; 
and (ii) process them once they are inside the computer. 
There is no point in acquiring data 
without some way to store it, 
so to start, let's get our hands dirty
with $n$-dimensional arrays, 
which we also call *tensors*.
If you already know the NumPy 
scientific computing package [@Harris.Millman.Walt.ea.2020], 
much of this will look familiar.
For all modern deep learning frameworks,
the *tensor class* (`ndarray` in MXNet, 
`Tensor` in PyTorch and TensorFlow,
`jax.Array` in JAX) 
resembles NumPy's `ndarray`,
with two additions.
First, the tensor class
supports automatic differentiation
(that section).
Second, it uses GPUs
to accelerate numerical computation,
whereas NumPy only runs on CPUs.
These properties make neural networks
both easy to code and fast to run.
(We cover how to place tensors on a GPU and move them between devices
in that section; until then, every tensor we create lives in
CPU memory.)


## Getting Started

To start, we import `tensorflow`. 
For brevity, practitioners 
often assign the alias `tf`.

In [ ]:
import tensorflow as tf

A tensor represents a (possibly multidimensional) array of numerical values.
In the one-dimensional case, i.e., when only one axis is needed for the data,
a tensor is called a *vector*.
With two axes, a tensor is called a *matrix*.
With $k > 2$ axes, we drop the specialized names
and just refer to the object as a $k^\textrm{th}$-*order tensor*.

Several functions create new tensors
prepopulated with values.
For example, by invoking `arange(n)`,
we can create a vector of evenly spaced values,
starting at 0 (included)
and ending at `n` (not included).
By default, the interval size is $1$.
Unless otherwise specified,
new tensors are stored in main memory
and designated for CPU-based computation.

In TensorFlow, this function is named `range` rather than `arange`.

In [ ]:
x = tf.range(12, dtype=tf.float32)
x

The `dtype` (data type) of a tensor determines how its elements are stored.
We use **32-bit** floating point (`float32`) as the default throughout these
chapters. Training systems also use lower-precision floating-point formats,
often together with higher-precision accumulation. that section
develops those formats and their numerical tradeoffs. You can inspect a tensor's type via its
`dtype` attribute. In finite precision, naive
computations can overflow or underflow, which is why later sections compute
quantities such as the softmax and cross-entropy with stabilized formulas
(that section).

Integer ranges like `range` default to an integer type,
which is why the cell above requests `dtype=tf.float32` explicitly.
To cast between types, use `tf.cast`.

In [ ]:
tf.size(x)

We can access a tensor's *shape* 
(the length along each axis)
by inspecting its `shape` attribute.
Because we are dealing with a vector here,
the `shape` contains just a single element
and is identical to the size.

In [ ]:
x.shape

We can change the shape of a tensor
without altering its size or values,
by invoking `reshape`.
For example, we can transform 
our vector `x` whose shape is (12,) 
to a matrix `X` with shape (3, 4).
This new tensor retains all elements
but reconfigures them into a matrix.
Notice that the elements of our vector
are laid out one row at a time and thus
`x[3] == X[0, 3]`, as the figure illustrates.

![Reshaping arranges the same 12 values into three rows without changing their order.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-reshape.svg)

In [ ]:
X = tf.reshape(x, (3, 4))
X

Note that specifying every shape component
to `reshape` is redundant.
Because we already know our tensor's size,
we can work out one component of the shape given the rest.
For example, given a tensor of size $n$
and target shape ($h$, $w$),
we know that $w = n/h$.
To automatically infer one component of the shape,
we can place a `-1` for the shape component
that should be inferred automatically.
In our case, instead of calling `x.reshape(3, 4)`,
we could have equivalently called `x.reshape(-1, 4)` or `x.reshape(3, -1)`.

For a contiguous tensor such as `x`, a framework can implement `reshape` by
changing only shape metadata. More generally, `reshape` may return either a
view that shares storage or a copy, depending on the layout and framework.
Code should therefore not rely on storage sharing unless it explicitly asks
for a view and satisfies that operation's layout requirements.

Practitioners often need to work with tensors
initialized to contain all 0s or 1s.
We can construct a tensor with all elements set to 0 
and a shape of (2, 3, 4) via the `zeros` function.

In [ ]:
tf.zeros((2, 3, 4))

Similarly, we can create a tensor 
with all 1s by invoking `ones`.

In [ ]:
tf.ones((2, 3, 4))

We often wish to 
sample each element randomly (and independently) 
from a given probability distribution.
For example, the parameters of neural networks
are often initialized randomly, in part to *break symmetry* between units:
if every weight in a layer started at the same value (say, zero),
the units would all compute the same thing and could never specialize.
The following snippet creates a tensor 
with elements drawn from 
a standard Gaussian (normal) distribution
with mean 0 and standard deviation 1.

In [ ]:
tf.random.normal(shape=[3, 4])

Random sampling raises the question of *reproducibility*. Calling a sampler
repeatedly gives different draws, which is usually what we want, but for
debugging or for figures that should not change between runs we fix a *seed*.

TensorFlow keeps a global generator that `tf.random.set_seed` resets:
after seeding, the sequence of draws is repeatable.

Finally, we can construct tensors by
supplying the exact values for each element
via (possibly nested) Python list(s)
containing numerical literals.
Here, we construct a matrix with a list of lists,
where the outermost list corresponds to axis 0,
and the inner list corresponds to axis 1.

In [ ]:
tf.constant([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

## Indexing and Slicing

As with Python lists,
we can access tensor elements 
by indexing (starting with 0).
To access an element based on its position
relative to the end of the list,
we can use negative indexing.
We can also access whole ranges of indices 
via slicing (e.g., `X[start:stop]`), 
where the returned value includes 
the first index (`start`) *but not the last* (`stop`).
Finally, when only one index (or slice)
is specified for a $k^\textrm{th}$-order tensor,
it is applied along axis 0.
Thus, in the following code,
`[-1]` selects the last row and `[1:3]`
selects the second and third rows.

In [ ]:
X[-1], X[1:3]

`Tensors` in TensorFlow are immutable, and cannot be assigned to.
`Variables` in TensorFlow are mutable containers of state that support
assignments. Keep in mind that gradients in TensorFlow do not flow backwards
through `Variable` assignments.

Beyond assigning a value to the entire `Variable`, we can write elements of a
`Variable` by specifying indices.

In [ ]:
X_var = tf.Variable(X)
X_var[1, 2].assign(17)
X_var

If we want to assign multiple elements the same value,
we apply the indexing on the left-hand side 
of the assignment operation.
For instance, `[:2, :]` accesses 
the first and second rows,
where `:` takes all the elements along axis 1 (column).
While we discussed indexing for matrices,
this also works for vectors
and for tensors of more than two dimensions.

In [ ]:
X_var = tf.Variable(X)
X_var[:2, :].assign(tf.ones(X_var[:2,:].shape, dtype=tf.float32) * 12)
X_var

## Operations

Now that we know how to construct tensors
and how to read from and write to their elements,
we can begin to manipulate them
with various mathematical operations.
Among the most useful of these 
are the *elementwise* operations.
These apply a standard scalar operation
to each element of a tensor.
For functions that take two tensors as inputs,
elementwise operations apply some standard binary operator
on each pair of corresponding elements.
We can create an elementwise function 
from any function that maps 
from a scalar to a scalar.

In mathematical notation, we denote such
*unary* scalar operators (taking one input)
by the signature 
$f: \mathbb{R} \rightarrow \mathbb{R}$.
This just means that the function maps
from any real number onto some other real number.
Most standard operators, including unary ones like $e^x$, can be applied elementwise.

In [ ]:
tf.exp(x)

Likewise, we denote *binary* scalar operators,
which map pairs of real numbers
to a (single) real number
via the signature 
$f: \mathbb{R}, \mathbb{R} \rightarrow \mathbb{R}$.
Given any two vectors $\mathbf{u}$ 
and $\mathbf{v}$ *of the same shape*,
and a binary operator $f$, we can produce a vector
$\mathbf{c} = F(\mathbf{u},\mathbf{v})$
by setting $c_i \gets f(u_i, v_i)$ for all $i$,
where $c_i, u_i$, and $v_i$ are the $i^\textrm{th}$ elements
of vectors $\mathbf{c}, \mathbf{u}$, and $\mathbf{v}$.
Here, we produced the vector-valued
$F: \mathbb{R}^d, \mathbb{R}^d \rightarrow \mathbb{R}^d$
by *lifting* the scalar function
to an elementwise vector operation.
The common standard arithmetic operators
for addition (`+`), subtraction (`-`), 
multiplication (`*`), division (`/`), 
and exponentiation (`**`)
have all been *lifted* to elementwise operations
for identically-shaped tensors of arbitrary shape.

In [ ]:
x = tf.constant([1.0, 2, 4, 8])
y = tf.constant([2.0, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

In addition to elementwise computations,
we can also perform linear algebraic operations,
such as dot products and matrix multiplications.
We will elaborate on these
in that section.

We can also *concatenate* multiple tensors,
stacking them end-to-end to form a larger one.
We just need to provide a list of tensors
and tell the system along which axis to concatenate.
The example below shows what happens when we concatenate
two matrices along rows (axis 0)
instead of columns (axis 1).
We can see that the first output's axis-0 length ($6$)
is the sum of the two input tensors' axis-0 lengths ($3 + 3$);
while the second output's axis-1 length ($8$)
is the sum of the two input tensors' axis-1 lengths ($4 + 4$).

In [ ]:
X = tf.reshape(tf.range(12, dtype=tf.float32), (3, 4))
Y = tf.constant([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
tf.concat([X, Y], axis=0), tf.concat([X, Y], axis=1)

Sometimes, we want to 
construct a boolean tensor via *logical statements*.
Take `X == Y` as an example.
For each position `i, j`, if `X[i, j]` and `Y[i, j]` are equal, 
then the corresponding entry in the result is `True`,
otherwise it is `False`.
(Booleans behave as 1 and 0 in arithmetic, so summing
such a mask counts the positions where the two tensors agree.)

In [ ]:
X == Y

Summing all the elements in the tensor yields a tensor with only one element.

In [ ]:
tf.reduce_sum(X)

## Broadcasting

By now, you know how to perform 
elementwise binary operations
on two tensors of the same shape. 
Under certain conditions,
even when shapes differ, 
we can still perform elementwise binary operations
by invoking the *broadcasting mechanism*
(a convention inherited from NumPy).

Broadcasting follows a simple rule. Line the two shapes up
from the *right* and compare them axis by axis: two axes are compatible when
they are equal or when one of them is $1$ (a missing leading axis counts as
$1$). The size-$1$ axis is then *stretched* to match the other, by virtually
copying its entries, and the elementwise operation is applied to the
resulting arrays. If any pair
of axes is incompatible, the operation raises an error rather than guessing.
the figure shows the mechanism at work on the example
that follows: each size-$1$ axis is stretched until both operands share the
shape $3\times2$.

![Broadcasting stretches the size-1 axes: a $3\times1$ column and a $1\times2$ row each expand to $3\times2$ before the elementwise addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-broadcasting.svg)

In [ ]:
a = tf.reshape(tf.range(3), (3, 1))
b = tf.reshape(tf.range(2), (1, 2))
a, b

Since `a` and `b` are $3\times1$ 
and $1\times2$ matrices, respectively,
their shapes do not match up.
Broadcasting produces a larger $3\times2$ matrix 
by replicating matrix `a` along the columns
and matrix `b` along the rows
before adding them elementwise.

In [ ]:
a + b

What happens when the shapes are *not* compatible? Lining up $(3, 2)$ and
$(2, 3)$ from the right pairs $2$ with $3$ and $3$ with $2$: neither axis pair
matches and neither member is $1$, so the framework refuses rather than
guessing.

In [ ]:
try:
    tf.ones((3, 2)) + tf.ones((2, 3))
except Exception as e:
    print(e)

## Saving Memory

Running operations can cause new memory to be
allocated to host results.
For example, if we write `Y = Y + X`,
we dereference the tensor that `Y` used to point to
and instead point `Y` at the newly allocated memory.
We can demonstrate the rebinding with Python's `id()` function,
which identifies a Python object for the duration of its lifetime.
Note that after we run `Y = Y + X`,
`id(Y)` changes.
That is because Python first evaluates `Y + X`,
allocating new memory for the result 
and then binds `Y` to the new tensor object. This test says nothing about the
address of the tensor's underlying storage.

In [ ]:
before = id(Y)
Y = Y + X
id(Y) == before

This might be undesirable for two reasons.
First, we do not want to run around
allocating memory unnecessarily all the time.
In machine learning, we often have
gigabytes of parameters
and update all of them multiple times per second.
In frameworks that support mutation, in-place updates can avoid these
allocations. They must be used with care because automatic differentiation
may need an earlier value to compute a gradient; functional and compiled
systems may instead reuse buffers internally without exposing mutation.
Second, we might point at the 
same parameters from multiple variables.
If we do not update in place, 
we must be careful to update all of these references,
lest we spring a memory leak 
or inadvertently refer to stale parameters.

`Variables` are mutable containers of state in TensorFlow. They provide
a way to store your model parameters.
We can assign the result of an operation
to a `Variable` with `assign`.
To illustrate this concept, 
we overwrite the values of `Variable` `Z`
after initializing it, using `zeros_like`,
to have the same shape as `Y`.

In [ ]:
Z = tf.Variable(tf.zeros_like(Y))
print('id(Z):', id(Z))
Z.assign(X + Y)
print('id(Z):', id(Z))

Because TensorFlow `Tensors` are immutable, there is no in-place assignment
for ordinary tensors; reuse a `Variable` (as above) when you need mutable
state. Compiling a computation with `tf.function` additionally lets
TensorFlow prune and reuse allocations for you; we return to graph
compilation and its performance benefits in that section.

## Conversion to Other Python Objects
<!-- d2l:prose id=ndarray-md-31 fw=mxnet, tensorflow -->

Converting to a NumPy tensor (`ndarray`), or vice versa, is easy.
The converted result does not share memory.
This minor inconvenience is actually quite important:
the framework may execute operations asynchronously,
possibly on a GPU, while NumPy works on a buffer in host memory.
If the two shared that buffer, each side would have to
synchronize with the other before reading or writing it;
copying removes the coordination.

In [ ]:
A = X.numpy()
B = tf.constant(A)
type(A), type(B)

To convert a size-1 tensor to a Python scalar,
we can invoke the `item` method or Python's built-in `float` and `int`.

In [ ]:
a = tf.constant([3.5]).numpy()
a, a.item(), float(a.item()), int(a.item())

## Discussion

The tensor class is the main interface for storing and manipulating data in deep learning libraries.
Tensors provide a variety of functionalities including construction routines; indexing and slicing; basic mathematics operations; broadcasting; memory-efficient assignment; and conversion to and from other Python objects.


## Exercises

1. Run the code in this section. Change the conditional statement `X == Y` to `X < Y` or `X > Y`, and then see what kind of tensor you can get.
1. Replace the two tensors that operate by element in the broadcasting mechanism with other shapes, e.g., 3-dimensional tensors. Is the result the same as expected?
1. Create the vector `x = arange(24)` and reshape it into a $(2, 3, 4)$ tensor using `-1` for one of the components. Which component does the framework infer, and why is at most one `-1` allowed?
1. Build a $(3, 4)$ tensor and predict, *before running*, the shape of its sum along `axis=0`, along `axis=1`, and with `keepdims=True`. Then check each prediction against the code.
1. Try to add two tensors whose shapes are *not* broadcast-compatible, for example shapes $(3, 2)$ and $(2, 3)$. What error do you get? Use the alignment rule from the broadcasting section to explain it, then find a reshape of one operand that makes the addition valid.
1. Recall the saving-memory discussion. Use `id()` to verify that `X[:] = X + Y` (or `X += Y`) keeps the same Python tensor object while `X = X + Y` binds `X` to a new object. Why does object identity alone not prove that the underlying storage was reused? Consult your framework's storage or buffer API and test storage identity where such an API exists.

[Discussions](https://d2l.discourse.group/t/187)